In [62]:
import pandas as pd

In [63]:
df = pd.read_csv('../Data/initial_data/PPR-ALL.csv', encoding='latin-1')


C:\Users\ash08\AppData\Local\Temp\ipykernel_12116\303652286.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/initial_data/PPR-ALL.csv', encoding='latin-1')


#### TO DO LIST
- check eircode, when was first eircode introduced: its random
- geocode address
- clean price, date of sale column: done
- check description column, could it be separated or not: done
- find a way to use VAT excl and not full market price. delete them if not needed: done
#### Possible datasets to join
- CSO Census 2022
- Pobal HP Deprivation Index
#### After cleaning
- compare same houses that were resold and for how much.
- do comparison by county
- get a way to create a map

In [64]:
### Checking Data here
#df[df['eircode'].notna()].sort_values(by = 'sale_date')
#df['desc'].unique()
#df[df['not_full_market_price'] == 'Yes']

In [65]:
df = df.rename(columns={'Date of Sale (dd/mm/yyyy)' : 'Sale_Date', 'Description of Property' : 'Desc', 'Price ()' : 'Price'})
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Convert 'sale_date' to a proper datetime object (format DD/MM/YYYY)
df['sale_date'] = pd.to_datetime(df['sale_date'], format='%d/%m/%Y')

# Clean the 'price' column by leaving only digits then convert to float
df['price'] = df['price'].astype(str).str.replace(r'[€€,]', '', regex=True).astype(float)

# Standardize text casing in 'county'
df['county'] = df['county'].str.title()


# Define the dictionary mapping all unique variants to your preferred labels
status_mapping = {
    'New Dwelling house /Apartment': 'New',
    'Teach/Árasán Cónaithe Nua': 'New',
    'Teach/?ras?n C?naithe Nua': 'New',
    'Second-Hand Dwelling house /Apartment': 'Second-Hand',
    'Teach/Árasán Cónaithe Atháimhe': 'Second-Hand'
}

# Apply the mapping to the description column
df['desc'] = df['desc'].replace(status_mapping)

# Drop houses that were not sold for market price to remove possible outliers
df = df[df['not_full_market_price'] == 'No']


# New Houses sometimes dont have price with VAT recorded. Its 13.5% for most data but from october 2025 for new apartments its 9%. 
# There is no column for type of property and it is not worth it for such small part of data getting it out of address. 
# Therefore, I will add 13.5% VAT to all new apartments/houses if VAT was not recorded in price

# df[df['vat_exclusive'] == 'Yes']['desc'].unique() # running this shows that only new builds dont have VAT accounted for.
VAT = 1.135   # 13.5%
df.loc[df['vat_exclusive'] == 'Yes', 'price'] = df['price'] * VAT
df['price'] = df['price'].astype(int)

# Drop columns that won't be used
df = df.drop(columns=['property_size_description', 'not_full_market_price', 'vat_exclusive'])

In [66]:
df

,sale_date,address,county,eircode,price,desc
0,2010-01-01,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,343000,Second-Hand
1,2010-01-03,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,209975,New
2,2010-01-04,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,438500,Second-Hand
3,2010-01-04,"1 The Haven, Mornington",Meath,NaN,400000,Second-Hand
4,2010-01-04,"11 Melville Heights, Kilkenny",Kilkenny,NaN,160000,Second-Hand
...,...,...,...,...,...,...
802619,2026-08-21,"No 24 The Drive, Stonehaven, Blessington Road",Kildare,NaN,524999,New
802620,2026-08-21,"No. 10 Furzefield Drive, Furzefield, Mooretown",Dublin,NaN,500000,New
802621,2026-08-21,"No. 3 Meadowbank Close, Millers Glen, Swords",Dublin,NaN,500000,New
802622,2026-08-21,"THE LOFT, ST PETERS SQ, WEXFORD",Wexford,Y35H567,140000,Second-Hand


#### GEOCODING



In [70]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize geocoder and rate limiter
geolocator = Nominatim(user_agent="irish_property_geocoder_v2")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

def build_search_query(row):
    address = str(row['address']).strip()
    eircode = str(row['eircode']).strip()
    
    if pd.notna(row['eircode']) and eircode.upper() not in ['NAN', 'UNKNOWN', 'NONE', '']:
        return f"{address}, {eircode}, Ireland"
    return f"{address}, Ireland"

# Custom wrapper function to print live progress
def geocode_with_print(query):
    print(f"Searching: {query}...", end=" ")
    result = geocode(query)
    
    if result:
        print(f"✅ Found ({result.latitude}, {result.longitude})")
    else:
        print("❌ Not found")
        
    return result

# 1. Build the search strings
df['search_query'] = df.apply(build_search_query, axis=1)

print(f"Starting geocoding for {len(df)} addresses. This will take roughly {len(df) * 1.1 / 60:.1f} minutes...\n")

# 2. Apply the custom printing function
df['location'] = df['search_query'].apply(geocode_with_print)

# 3. Extract coordinates
df['latitude'] = df['location'].apply(lambda loc: loc.latitude if loc else None)
df['longitude'] = df['location'].apply(lambda loc: loc.longitude if loc else None)

# Clean up
df = df.drop(columns=['search_query', 'location'])
print("\nGeocoding complete!")
print(df[['address', 'latitude', 'longitude']].head())

KeyboardInterrupt: 